Perfect 👍 You want a **complete end-to-end Python project scaffold** using **LangGraph** with **multiple agents** (weather, pollution, parent orchestrator) and proper modular structure.

I’ll scaffold it like a **real-world project**:

---

## 📂 Project Structure

```
multi_agent_project/
│── main.py
│── config/
│   ├── settings.py
│   └── __init__.py
│
│── agents/
│   ├── agent_factory.py
│   ├── parent_agent.py
│   ├── weather_agent.py
│   ├── pollution_agent.py
│   └── __init__.py
│
│── tools/
│   ├── weather_tools.py
│   ├── pollution_tools.py
│   └── __init__.py
│
│── mcp_servers/
│   ├── weather_mcp.py
│   ├── pollution_mcp.py
│   └── __init__.py
│
│── requirements.txt
│── .env
```

---

## 🔑 `config/settings.py`

```python
import os
from dotenv import load_dotenv

# Load environment variables
dotenv_path = os.path.join(os.path.dirname(__file__), "../.env")
load_dotenv(dotenv_path)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# Default models
OPENAI_MODEL = "gpt-4o"
GEMINI_MODEL = "gemini-2.0-flash"
```

---

## ⚡ Tools

### `tools/weather_tools.py`

```python
from langchain_core.tools import tool

@tool
def get_city_weather(city: str) -> str:
    """Fetch weather for a city."""
    if city.lower() == "mumbai":
        return "Rainy"
    elif city.lower() == "delhi":
        return "Sunny"
    return "Moderate"

@tool
def get_country_weather(country: str) -> str:
    """Fetch average weather for a country."""
    if country.lower() == "india":
        return "Hot"
    return "Mild"
```

### `tools/pollution_tools.py`

```python
from langchain_core.tools import tool

@tool
def get_city_pollution(city: str) -> str:
    """Fetch AQI for a city."""
    if city.lower() == "delhi":
        return "High Pollution"
    elif city.lower() == "mumbai":
        return "Moderate Pollution"
    return "Clean"

@tool
def get_country_pollution(country: str) -> str:
    """Fetch average AQI for a country."""
    if country.lower() == "india":
        return "Severe Pollution"
    return "Normal"
```

---

## 🤖 Agents

### `agents/weather_agent.py`

```python
from langchain_openai import ChatOpenAI
from tools.weather_tools import get_city_weather, get_country_weather
from config.settings import OPENAI_MODEL

def create_weather_agent():
    llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0)
    return llm.bind_tools([get_city_weather, get_country_weather])
```

### `agents/pollution_agent.py`

```python
from langchain_google_genai import ChatGoogleGenerativeAI
from tools.pollution_tools import get_city_pollution, get_country_pollution
from config.settings import GEMINI_MODEL

def create_pollution_agent():
    llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0)
    return llm.bind_tools([get_city_pollution, get_country_pollution])
```

### `agents/parent_agent.py`

```python
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from agents.weather_agent import create_weather_agent
from agents.pollution_agent import create_pollution_agent
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

def create_parent_agent():
    weather_agent = create_weather_agent()
    pollution_agent = create_pollution_agent()

    parent_llm = ChatOpenAI(model="gpt-4o", temperature=0)

    def router_node(state: State):
        user_input = state["messages"][-1].content.lower()
        if "weather" in user_input:
            msg = weather_agent.invoke(state["messages"])
        elif "pollution" in user_input:
            msg = pollution_agent.invoke(state["messages"])
        else:
            msg = parent_llm.invoke(state["messages"])
        return {"messages": [msg]}

    builder = StateGraph(State)
    builder.add_node("router", router_node)
    builder.add_edge(START, "router")
    builder.add_edge("router", END)

    return builder.compile()
```

---

## 🏭 `agents/agent_factory.py`

```python
from agents.parent_agent import create_parent_agent
from agents.weather_agent import create_weather_agent
from agents.pollution_agent import create_pollution_agent

def get_agent(agent_name: str):
    if agent_name == "parent":
        return create_parent_agent()
    elif agent_name == "weather":
        return create_weather_agent()
    elif agent_name == "pollution":
        return create_pollution_agent()
    else:
        raise ValueError(f"Unknown agent: {agent_name}")
```

---

## 🎯 `main.py`

```python
from agents.agent_factory import get_agent
from langchain_core.messages import HumanMessage

if __name__ == "__main__":
    graph = get_agent("parent")

    user_queries = [
        "What is the weather in Mumbai?",
        "How is the pollution in Delhi?",
        "Tell me about India’s climate"
    ]

    for query in user_queries:
        print(f"\nUser: {query}")
        result = graph.invoke({"messages": [HumanMessage(content=query)]})
        print("Bot:", result["messages"][-1].content)
```

---

## ⚙️ `requirements.txt`

```
langchain
langgraph
langchain-openai
langchain-google-genai
python-dotenv
```

---

## 🚀 Run the Project

```bash
cd multi_agent_project
pip install -r requirements.txt
python main.py
```

---

✅ This project supports:

* **Multiple agents** (Weather / Pollution / Parent Router).
* **Conditional routing** inside parent agent.
* **Modular structure** (config, tools, agents, mcp).
* **LLM selection (OpenAI / Gemini)** based on config.
* **Future ready** → can add **RAG / ReAct / Chain-of-Thought**.

---

Do you want me to **extend the parent agent with ReAct reasoning** (chain-of-thought + intermediate tool calls) so it becomes a **smarter orchestrator** instead of just a simple router?
